# Phase 2 — Spark Fundamentals

**Objectifs:**
- [x] SparkSession
- [x] DataFrames
- [x] Transformations / Actions
- [x] Partitions
- [x] Jobs / Stages / Tasks
- [x] Shuffle
- [x] Cache / Persist

**Livrable:** Notebook de pratique Spark

**Données:** NYC Yellow Taxi — janvier 2024 (2 964 624 courses) + 265 zones, lues depuis le Volume `raw/landing` (Phase 1).

## 0. Setup

In [1]:
from lakehouse_config import *
spark = get_spark()
setup_schemas(spark)

from pyspark.sql import functions as F
from pyspark.sql.types import *

import io, time
from contextlib import redirect_stdout


def timed(label, fn):
    """Execute fn() et affiche le temps d'execution."""
    start = time.perf_counter()
    result = fn()
    print(f"{label}: {time.perf_counter() - start:.2f} s")
    return result


def plan_of(df):
    """Retourne le plan physique d'un DataFrame sous forme de texte."""
    buf = io.StringIO()
    with redirect_stdout(buf):
        df.explain()
    return buf.getvalue()


def count_shuffles(df):
    """Nombre de shuffles dans le plan (Photon: ShuffleExchangeSink, Spark classique: Exchange)."""
    plan = plan_of(df)
    return plan.count("ShuffleExchangeSink") or plan.count("Exchange ")

## 1. SparkSession

La **SparkSession** est le point d'entrée unique de Spark: lire des données, créer des DataFrames, exécuter du SQL, accéder au catalogue.

Ici on n'utilise pas `SparkSession.builder.master("local[*]")`: le code tourne dans VS Code mais **les calculs s'exécutent sur Databricks Serverless** via **Databricks Connect** (Spark Connect). `get_spark()` (dans `lakehouse_config.py`) crée cette session distante.

In [2]:
print("Spark version :", spark.version)
print("Type de session:", type(spark))

Spark version : 4.2.0
Type de session: <class 'pyspark.sql.connect.session.SparkSession'>


In [3]:
spark.sql("SELECT current_user() AS user, current_catalog() AS catalog, current_schema() AS schema").show(truncate=False)

+--------------------------+---------+-------+
|user                      |catalog  |schema |
+--------------------------+---------+-------+
|marwane2001morfi@gmail.com|workspace|default|
+--------------------------+---------+-------+



In [4]:
print("Catalogs:", [c.name for c in spark.catalog.listCatalogs()])
spark.sql("SHOW SCHEMAS IN workspace").show()

Catalogs: ['samples', 'system', 'workspace']


+------------------+
|      databaseName|
+------------------+
|            bronze|
|           default|
|              gold|
|information_schema|
|               ops|
|               raw|
|            silver|
+------------------+



In [5]:
# Tables creees en Phase 1
spark.sql("SHOW TABLES IN workspace.ops").show(truncate=False)

+--------+---------------+-----------+
|database|tableName      |isTemporary|
+--------+---------------+-----------+
|ops     |data_dictionary|false      |
+--------+---------------+-----------+



In [6]:
# Configuration: certaines options sont lisibles, mais la plupart ne sont pas modifiables en Serverless
for key in ["spark.sql.shuffle.partitions", "spark.sql.adaptive.enabled", "spark.app.name"]:
    try:
        print(f"{key} = {spark.conf.get(key)}")
    except Exception as e:
        print(f"{key} -> non disponible ({type(e).__name__})")

spark.sql.shuffle.partitions = auto


spark.sql.adaptive.enabled -> non disponible (AnalysisException)


spark.app.name -> non disponible (AnalysisException)


### Observations — SparkSession
- La session est une session **Spark Connect** distante: le client (VS Code) envoie des plans, le cluster Serverless les exécute.
- En **Serverless**, Databricks gère la session: `.appName(...)` et la plupart des `spark.conf.set(...)` sont ignorés ou refusés (`CONFIG_NOT_AVAILABLE`).
- Convention: la session s'appelle **toujours `spark`** (ne jamais nommer une variable `SparkSession`, c'est le nom de la classe).

## 2. DataFrames

Un **DataFrame** = une table distribuée (lignes + colonnes typées) répartie sur plusieurs machines.
Différence avec pandas: pandas tient sur **une seule machine** en mémoire, Spark **répartit** les données et les calculs.

### 2.1 Créer un DataFrame

In [7]:
# A) Depuis des fichiers (Volume Databricks)
trips = spark.read.parquet(f"{RAW_VOLUME}/yellow_tripdata_2024-01.parquet")
zones = spark.read.csv(f"{RAW_VOLUME}/{ZONES_FILE}", header=True, inferSchema=True)

print(f"trips: {trips.count():,} lignes x {len(trips.columns)} colonnes")
print(f"zones: {zones.count():,} lignes x {len(zones.columns)} colonnes")

trips: 2,964,624 lignes x 19 colonnes


zones: 265 lignes x 4 colonnes


In [8]:
# B) A la main, depuis une liste Python (utile pour les tests)
villes = spark.createDataFrame(
    [(1, "Casablanca", 3.75), (2, "Rabat", 0.58), (3, "Marrakech", 0.93), (4, "Tanger", 0.95)],
    "id int, ville string, population_millions double",
)
villes.printSchema()
villes.show()

root
 |-- id: integer (nullable = true)
 |-- ville: string (nullable = true)
 |-- population_millions: double (nullable = true)



+---+----------+-------------------+
| id|     ville|population_millions|
+---+----------+-------------------+
|  1|Casablanca|               3.75|
|  2|     Rabat|               0.58|
|  3| Marrakech|               0.93|
|  4|    Tanger|               0.95|
+---+----------+-------------------+



### 2.2 Les opérations de base

In [9]:
# select: choisir des colonnes
trips.select("tpep_pickup_datetime", "trip_distance", "total_amount").show(5)

+--------------------+-------------+------------+
|tpep_pickup_datetime|trip_distance|total_amount|
+--------------------+-------------+------------+
| 2024-01-01 00:57:55|         1.72|        22.7|
| 2024-01-01 00:03:00|          1.8|       18.75|
| 2024-01-01 00:17:06|          4.7|        31.3|
| 2024-01-01 00:36:38|          1.4|        17.0|
| 2024-01-01 00:46:51|          0.8|        16.1|
+--------------------+-------------+------------+
only showing top 5 rows


In [10]:
# filter / where: filtrer des lignes
long_trips = trips.filter(F.col("trip_distance") > 10)
print(f"Courses > 10 miles: {long_trips.count():,}")

Courses > 10 miles: 228,254


In [11]:
# withColumn: ajouter une colonne calculee | withColumnRenamed: renommer | drop: supprimer
trips_enriched = (
    trips
    .withColumn("duration_min",
                F.round((F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60, 1))
    .withColumnRenamed("VendorID", "vendor_id")
    .drop("store_and_fwd_flag")
)
trips_enriched.select("vendor_id", "trip_distance", "duration_min", "total_amount").show(5)

+---------+-------------+------------+------------+
|vendor_id|trip_distance|duration_min|total_amount|
+---------+-------------+------------+------------+
|        2|         1.72|        19.8|        22.7|
|        1|          1.8|         6.6|       18.75|
|        1|          4.7|        17.9|        31.3|
|        1|          1.4|         8.3|        17.0|
|        1|          0.8|         6.1|        16.1|
+---------+-------------+------------+------------+
only showing top 5 rows


In [12]:
# orderBy: trier -> top 5 des courses les plus cheres
trips.select("tpep_pickup_datetime", "trip_distance", "total_amount") \
     .orderBy(F.desc("total_amount")) \
     .show(5)

+--------------------+-------------+------------+
|tpep_pickup_datetime|trip_distance|total_amount|
+--------------------+-------------+------------+
| 2024-01-20 11:18:47|          0.0|      5000.0|
| 2024-01-20 11:19:33|          0.0|      5000.0|
| 2024-01-20 11:27:48|          0.0|      2500.0|
| 2024-01-24 13:44:43|          0.0|      2500.0|
| 2024-01-20 11:20:15|          0.0|      2500.0|
+--------------------+-------------+------------+
only showing top 5 rows


In [13]:
# groupBy + agg: agregations par type de paiement
(trips.groupBy("payment_type")
      .agg(F.count("*").alias("nb_courses"),
           F.round(F.avg("total_amount"), 2).alias("prix_moyen"),
           F.round(F.avg("tip_amount"), 2).alias("pourboire_moyen"))
      .orderBy("payment_type")
      .show())

+------------+----------+----------+---------------+
|payment_type|nb_courses|prix_moyen|pourboire_moyen|
+------------+----------+----------+---------------+
|           0|    140162|     25.81|           1.55|
|           1|   2319046|     28.26|           4.17|
|           2|    439191|     22.88|            0.0|
|           3|     19597|      8.76|           0.01|
|           4|     46628|      1.77|           0.04|
+------------+----------+----------+---------------+



In [14]:
# join + chaining: top 5 des zones de prise en charge (avec leur nom)
top_pickup_zones = (
    trips
    .join(zones, trips.PULocationID == zones.LocationID, "left")
    .groupBy("Borough", "Zone")
    .agg(F.count("*").alias("nb_courses"))
    .orderBy(F.desc("nb_courses"))
    .limit(5)
)
top_pickup_zones.show(truncate=False)

+---------+---------------------+----------+
|Borough  |Zone                 |nb_courses|
+---------+---------------------+----------+
|Queens   |JFK Airport          |145240    |
|Manhattan|Midtown Center       |143471    |
|Manhattan|Upper East Side South|142708    |
|Manhattan|Upper East Side North|136465    |
|Manhattan|Midtown East         |106717    |
+---------+---------------------+----------+



### 2.3 La même requête en SQL

In [15]:
trips.createOrReplaceTempView("trips")
zones.createOrReplaceTempView("zones")

spark.sql("""
    SELECT z.Borough, z.Zone, COUNT(*) AS nb_courses
    FROM trips t
    LEFT JOIN zones z ON t.PULocationID = z.LocationID
    GROUP BY z.Borough, z.Zone
    ORDER BY nb_courses DESC
    LIMIT 5
""").show(truncate=False)

+---------+---------------------+----------+
|Borough  |Zone                 |nb_courses|
+---------+---------------------+----------+
|Queens   |JFK Airport          |145240    |
|Manhattan|Midtown Center       |143471    |
|Manhattan|Upper East Side South|142708    |
|Manhattan|Upper East Side North|136465    |
|Manhattan|Midtown East         |106717    |
+---------+---------------------+----------+



### 2.4 Immutabilité

In [16]:
# Un DataFrame ne change jamais: chaque operation retourne un NOUVEAU DataFrame
filtered = trips.filter(F.col("trip_distance") > 10)
print("trips   :", f"{trips.count():,}", "lignes (inchange)")
print("filtered:", f"{filtered.count():,}", "lignes (nouveau DataFrame)")

trips   : 2,964,624 lignes (inchange)


filtered: 228,254 lignes (nouveau DataFrame)


### Observations — DataFrames
- On crée un DataFrame depuis un **fichier** (`spark.read`) ou depuis une **liste Python** (`createDataFrame`).
- Opérations de base: `select`, `filter`, `withColumn`, `withColumnRenamed`, `drop`, `orderBy`, `groupBy().agg()`, `join`.
- **API Python et SQL donnent exactement le même résultat** (même moteur, même plan).
- Les DataFrames sont **immuables**: `trips` n'est jamais modifié, on assigne le résultat à une nouvelle variable.
- Résultat métier: **JFK Airport** et **Upper East Side** sont parmi les zones de prise en charge les plus actives.

## 3. Transformations / Actions

- **Transformation** (`select`, `filter`, `withColumn`, `groupBy`, `join`…) → **lazy**: Spark n'exécute rien, il construit un **plan**.
- **Action** (`show`, `count`, `collect`, `write`, `toPandas`…) → déclenche **l'exécution** du plan.

Avantage: Spark voit tout le plan avant d'exécuter et peut l'**optimiser** (Catalyst).

In [17]:
# 1) Chaine de transformations -> instantane, rien n'est calcule
pipeline = timed("Transformations (lazy)", lambda: (
    trips
    .filter(F.col("trip_distance") > 0)
    .withColumn("duration_min", (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60)
    .filter(F.col("duration_min").between(1, 180))
    .groupBy("PULocationID")
    .agg(F.avg("total_amount").alias("prix_moyen"))
))

# 2) Action -> Spark execute tout le plan
n = timed("Action count()", lambda: pipeline.count())
print(f"Resultat: {n} zones")

Transformations (lazy): 0.00 s


Action count(): 0.52 s
Resultat: 257 zones


In [18]:
# Le plan que Spark va executer (optimise par Catalyst)
pipeline.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonGroupingAgg(keys=[PULocationID#14501], functions=[finalmerge_avg(merge sum#14776, count#14777L) AS avg(total_amount)#14773])
         +- PhotonShuffleExchangeSource false
            +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#20097]
               +- PhotonShuffleExchangeSink hashpartitioning(PULocationID#14501, 47)
                  +- PhotonGroupingAgg(keys=[PULocationID#14501], functions=[partial_avg(total_amount#14510) AS (sum#14776, count#14777L)])
                     +- PhotonProject [PULocationID#14501, total_amount#14510]
                        +- PhotonScan parquet [tpep_pickup_datetime#14495,tpep_dropoff_datetime#14496,trip_distance#14498,PULocationID#14501,total_amount#14510] DataFilters: [isnotnull(trip_distance#14498), (trip_distance#14498 > 0.0), ((cast((unix_timestamp(tpep_dropoff..., DictionaryFilters: [(trip_distance#1449

### Transformations narrow vs wide

| Type | Exemples | Shuffle ? | Coût |
|---|---|---|---|
| **Narrow** | `select`, `filter`, `withColumn`, `drop` | ❌ Non — chaque partition est traitée seule | Faible |
| **Wide** | `groupBy`, `join`, `orderBy`, `distinct`, `repartition` | ✅ Oui — les données bougent entre partitions | Élevé |

### Observations — Transformations / Actions
- Les transformations ne lisent **aucune donnée**: Spark enregistre seulement le plan (le petit temps mesuré = aller-retour réseau pour valider le schéma via Spark Connect).
- Les 3 millions de lignes sont lues et calculées **au moment de l'action** (`count`).
- Dans le plan, les deux `filter` sont **fusionnés** et poussés au plus près de la lecture (*predicate pushdown*): c'est l'optimiseur Catalyst.

## 4. Partitions

Spark découpe un DataFrame en **partitions** (morceaux). Chaque partition est traitée par **une task** en parallèle.
- Trop peu de partitions → peu de parallélisme.
- Trop de partitions → beaucoup de petites tasks (overhead).

In [19]:
def partition_sizes(df):
    """Nombre de lignes par partition."""
    return (df.withColumn("partition_id", F.spark_partition_id())
              .groupBy("partition_id").count()
              .orderBy("partition_id"))


p = partition_sizes(trips)
print(f"Partitions a la lecture: {p.count()}")
p.show()

Partitions a la lecture: 1


+------------+-------+
|partition_id|  count|
+------------+-------+
|           0|2964624|
+------------+-------+



In [20]:
# repartition(n): redistribue en n partitions EQUILIBREES (provoque un shuffle)
p8 = partition_sizes(trips.repartition(8))
print(f"Apres repartition(8): {p8.count()} partitions")
p8.show()

Apres repartition(8): 8 partitions


+------------+------+
|partition_id| count|
+------------+------+
|           0|370578|
|           1|370578|
|           2|370578|
|           3|370578|
|           4|370578|
|           5|370578|
|           6|370578|
|           7|370578|
+------------+------+



In [21]:
# coalesce(n): REDUIT le nombre de partitions en fusionnant (pas de shuffle complet)
p2 = partition_sizes(trips.repartition(8).coalesce(2))
print(f"Apres coalesce(2): {p2.count()} partitions")
p2.show()

Apres coalesce(2): 2 partitions


+------------+-------+
|partition_id|  count|
+------------+-------+
|           0|1482312|
|           1|1482312|
+------------+-------+



In [22]:
# repartition par colonne: toutes les lignes d'une meme valeur dans la meme partition
by_vendor = partition_sizes(trips.repartition(4, "VendorID"))
by_vendor.show()
trips.repartition(4, "VendorID").withColumn("partition_id", F.spark_partition_id()) \
     .groupBy("partition_id", "VendorID").count().orderBy("partition_id").show()

+------------+-------+
|partition_id|  count|
+------------+-------+
|           1|    260|
|           2|2234632|
|           3| 729732|
+------------+-------+



+------------+--------+-------+
|partition_id|VendorID|  count|
+------------+--------+-------+
|           1|       6|    260|
|           2|       2|2234632|
|           3|       1| 729732|
+------------+--------+-------+



In [23]:
# Le plan montre la difference: repartition -> Exchange (shuffle), coalesce -> pas d'Exchange
print("repartition(8) -> shuffles:", count_shuffles(trips.repartition(8)))
print("coalesce(2)    -> shuffles:", count_shuffles(trips.coalesce(2)))

repartition(8) -> shuffles: 1


coalesce(2)    -> shuffles: 0


### Observations — Partitions

| | `repartition(n)` | `coalesce(n)` |
|---|---|---|
| Augmenter les partitions | ✅ | ❌ |
| Réduire les partitions | ✅ | ✅ |
| Shuffle | ✅ Oui (coûteux) | ❌ Non |
| Partitions équilibrées | ✅ | ❌ Pas forcément |
| Usage typique | Équilibrer avant un calcul lourd / partitionner par clé | Réduire le nombre de fichiers avant une écriture |

- **À la lecture: 1 seule partition.** Le fichier fait ~50 Mo, moins que la taille max d'une partition de lecture (`spark.sql.files.maxPartitionBytes` = 128 Mo par défaut) → Spark ne le découpe pas. Sur des fichiers plus gros (ou plusieurs mois), il y aurait plusieurs partitions.
- `repartition(8)` → 8 partitions **équilibrées** (~370 000 lignes chacune), au prix d'**un shuffle**.
- `coalesce(2)` → **0 shuffle**, mais les partitions ne sont pas forcément équilibrées.
- `repartition(4, "VendorID")` regroupe chaque vendor dans **une seule partition** → partitions **déséquilibrées** (VendorID 2 = 75 % des données): c'est le **data skew**.

## 5. Jobs / Stages / Tasks

| Niveau | Définition | Analogie (construire une maison) |
|---|---|---|
| **Job** | Créé par **chaque action** (`count`, `show`, `write`…) | La maison entière |
| **Stage** | Partie d'un job **entre deux shuffles** | Fondations → murs → toit |
| **Task** | Travail d'**une partition** dans un stage | Un ouvrier qui monte un mur |

**Règle:** `nb_stages ≈ nb_shuffles (Exchange) + 1` et `nb_tasks d'un stage = nb_partitions`.

In [24]:
queries = {
    "filter + count (narrow)": trips.filter(F.col("trip_distance") > 5),
    "groupBy (1 shuffle)": trips.groupBy("payment_type").count(),
    "groupBy + orderBy (2 shuffles)": trips.groupBy("PULocationID").count().orderBy(F.desc("count")),
    "join + groupBy + orderBy": trips.join(zones, trips.PULocationID == zones.LocationID)
                                     .groupBy("Borough").count().orderBy(F.desc("count")),
}

for name, df in queries.items():
    shuffles = count_shuffles(df)
    print(f"{name:35s} -> {shuffles} shuffle(s) -> ~{shuffles + 1} stage(s)")

filter + count (narrow)             -> 0 shuffle(s) -> ~1 stage(s)


groupBy (1 shuffle)                 -> 1 shuffle(s) -> ~2 stage(s)


groupBy + orderBy (2 shuffles)      -> 2 shuffle(s) -> ~3 stage(s)


join + groupBy + orderBy            -> 3 shuffle(s) -> ~4 stage(s)


### Voir les jobs dans Databricks
En Serverless il n'y a pas de Spark UI classique. Pour voir l'exécution réelle:
1. Exécuter une requête (ex: le `groupBy` ci-dessus) depuis un notebook Databricks.
2. Sidebar → **Query History** → cliquer sur la requête → **Query Profile**.
3. On y voit les **stages**, le nombre de **tasks**, les lignes lues et le temps de chaque opérateur.

### Observations — Jobs / Stages / Tasks
- Chaque **action** = un **job**. Un `count()` = 1 job.
- Une requête **sans shuffle** (filter) tient en **1 stage**.
- Chaque **shuffle** (`groupBy`, `orderBy`, `join`) **coupe** le job en un stage supplémentaire.
- **Note sur la jointure:** `explain()` montre le plan **initial** (avant exécution). Il prévoit un shuffle pour la jointure; à l'exécution, **AQE** (Adaptive Query Execution) peut la convertir en **broadcast join** car `zones` est petite. On peut aussi le forcer avec `F.broadcast(zones)` (voir section 6).

## 6. Shuffle

Le **shuffle** = redistribution des données **entre les partitions/machines** (écriture disque + réseau).
C'est l'opération **la plus coûteuse** de Spark. Dans un plan, il apparaît sous le nom **`Exchange`** (Spark classique) ou **`PhotonShuffleExchangeSink / Source`** (moteur Photon utilisé par Databricks Serverless).

In [25]:
print("=== filter (narrow) ===")
trips.filter(F.col("trip_distance") > 5).explain()

=== filter (narrow) ===
== Physical Plan ==
PhotonResultStage
+- PhotonColumnarToRow
   +- PhotonScan parquet [VendorID#14494,tpep_pickup_datetime#14495,tpep_dropoff_datetime#14496,passenger_count#14497L,trip_distance#14498,RatecodeID#14499L,store_and_fwd_flag#14500,PULocationID#14501,DOLocationID#14502,payment_type#14503L,fare_amount#14504,extra#14505,mta_tax#14506,tip_amount#14507,tolls_amount#14508,improvement_surcharge#14509,total_amount#14510,congestion_surcharge#14511,Airport_fee#14512] DataFilters: [isnotnull(trip_distance#14498), (trip_distance#14498 > 5.0)], DictionaryFilters: [(trip_distance#14498 > 5.0)], Format: parquet, Location: InMemoryFileIndex(1 paths)[dbfs:/Volumes/workspace/raw/landing/yellow_tripdata_2024-01.parquet], OptionalDataFilters: [], PartitionFilters: [], ReadSchema: struct<VendorID:int,tpep_pickup_datetime:timestamp_ntz,tpep_dropoff_datetime:timestamp_ntz,passen..., RequiredDataFilters: [isnotnull(trip_distance#14498), (trip_distance#14498 > 5.0)]


== Pho

In [26]:
print("=== groupBy (wide) ===")
trips.groupBy("payment_type").count().explain()

=== groupBy (wide) ===


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonGroupingAgg(keys=[payment_type#14503L], functions=[finalmerge_count(merge count#14943L) AS count(1)#14941L])
         +- PhotonShuffleExchangeSource false
            +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#22084]
               +- PhotonShuffleExchangeSink hashpartitioning(payment_type#14503L, 47)
                  +- PhotonGroupingAgg(keys=[payment_type#14503L], functions=[partial_count(1) AS count#14943L])
                     +- PhotonScan parquet [payment_type#14503L] DataFilters: [], DictionaryFilters: [], Format: parquet, Location: InMemoryFileIndex(1 paths)[dbfs:/Volumes/workspace/raw/landing/yellow_tripdata_2024-01.parquet], OptionalDataFilters: [], PartitionFilters: [], ReadSchema: struct<payment_type:bigint>, RequiredDataFilters: []


== Photon Explanation ==
The query is fully supported by Photon.


In [27]:
# Comparaison des temps: narrow vs wide
timed("filter + count        ", lambda: trips.filter(F.col("trip_distance") > 5).count())
timed("groupBy (shuffle)     ", lambda: trips.groupBy("PULocationID", "DOLocationID").count().count())
timed("distinct (shuffle)    ", lambda: trips.select("PULocationID", "DOLocationID", "payment_type").distinct().count())

filter + count        : 0.82 s


groupBy (shuffle)     : 0.45 s


distinct (shuffle)    : 0.51 s


52350

### Éviter le shuffle d'une jointure: le broadcast join
Quand une table est **petite** (`zones` = 265 lignes), Spark l'envoie **en entier à chaque machine** (*broadcast*) → la grande table (`trips`) n'a pas besoin d'être shufflée.

In [28]:
broadcast_plan = plan_of(trips.join(F.broadcast(zones), trips.PULocationID == zones.LocationID))
print("BroadcastHashJoin" in broadcast_plan and "-> Broadcast join: zones est copiee sur chaque machine, trips n'est pas shufflee")
print([line.strip() for line in broadcast_plan.splitlines() if "Join" in line or "Broadcast" in line][:4])

-> Broadcast join: zones est copiee sur chaque machine, trips n'est pas shufflee
['+- PhotonBroadcastHashJoin [PULocationID#14501], [LocationID#14537], Inner, BuildRight, false, true, false']


### Observations — Shuffle
- `filter` → **aucun shuffle** dans le plan. `groupBy` → **`ShuffleExchangeSink hashpartitioning(payment_type, ...)`**: les lignes de même clé sont envoyées vers la même partition.
- Le `groupBy` fait d'abord une **agrégation partielle** dans chaque partition (`partial_count` **avant** le shuffle, `finalmerge_count` **après**) → on envoie quelques lignes par partition au lieu de 3 millions.
- **Sur les temps:** avec seulement ~50 Mo, la différence narrow / wide est **négligeable** (< 1 s, dominée par l'aller-retour réseau VS Code ↔ Databricks). Le coût du shuffle devient visible sur des **gros volumes** (plusieurs Go): c'est pour ça qu'on raisonne sur le **plan**, pas seulement sur le chrono.
- **Bonnes pratiques:** filtrer **avant** de joindre/agréger, **broadcaster** les petites tables, éviter les `orderBy` / `distinct` inutiles.

## 7. Cache / Persist

`cache()` / `persist()` gardent un DataFrame **en mémoire (et/ou disque)** après la première action, pour ne pas le recalculer à chaque réutilisation.

| Méthode | Stockage |
|---|---|
| `df.cache()` | `MEMORY_AND_DISK` (par défaut pour les DataFrames) |
| `df.persist(StorageLevel.MEMORY_ONLY)` | Mémoire uniquement (recalcule ce qui ne tient pas) |
| `df.persist(StorageLevel.DISK_ONLY)` | Disque uniquement |
| `df.unpersist()` | Libère le cache |

⚠️ **Le cache est lazy:** il n'est rempli qu'à la **première action**.

In [29]:
from pyspark import StorageLevel

clean_trips = trips.filter((F.col("trip_distance") > 0) & (F.col("total_amount") > 0))

print("Storage level avant cache:", clean_trips.storageLevel)


def three_actions(df):
    df.count()
    df.agg(F.avg("total_amount")).collect()
    df.groupBy("payment_type").count().collect()


# 1) SANS cache: chaque action relit le fichier et refait le filtre
timed("3 actions SANS cache", lambda: three_actions(clean_trips))

Storage level avant cache: Serialized 1x Replicated


3 actions SANS cache: 1.95 s


In [30]:
# 2) AVEC cache
clean_trips.cache()
print("Storage level apres cache():", clean_trips.storageLevel)

timed("1ere action (remplit le cache)", lambda: clean_trips.count())
timed("3 actions AVEC cache          ", lambda: three_actions(clean_trips))

Storage level apres cache(): Disk Serialized 1x Replicated


1ere action (remplit le cache): 0.62 s


3 actions AVEC cache          : 2.71 s


In [31]:
# 3) Toujours liberer le cache quand on n'en a plus besoin
clean_trips.unpersist()
print("Storage level apres unpersist():", clean_trips.storageLevel)

Storage level apres unpersist(): Serialized 1x Replicated


### Observations — Cache / Persist
- **Serverless:** `cache()` nécessite **Databricks Connect ≥ 18.1.8** (avec la version 16 on obtient `CACHE_REQUIRES_NEWER_CLIENT_ON_SERVERLESS_GC`). Le niveau de stockage est **`Disk Serialized`**: en Serverless le cache est sur **disque local SSD**, pas en RAM.
- `cache()` est **lazy**: la 1ère action paie le calcul **et** le remplissage du cache; les suivantes relisent le cache.
- **Résultat mesuré: le cache n'a PAS accéléré** les 3 actions ici (même un peu plus lent). Raisons: les données sont petites (~50 Mo), le filtre est très simple, Databricks garde déjà les fichiers Parquet en *disk cache*, et le temps est dominé par l'aller-retour réseau VS Code ↔ Databricks. **Le cache a un coût** (écriture + lecture du cache) qui n'est rentable que si le recalcul coûte plus cher.
- Le gain devient important quand le DataFrame est **coûteux à calculer** (jointures, agrégations lourdes, gros volumes) et réutilisé → **toujours mesurer** avant de cacher.
- **Quand cacher:** un DataFrame **coûteux** à calculer et **réutilisé plusieurs fois** (ex: données nettoyées utilisées par 5 agrégations).
- **Quand ne pas cacher:** un DataFrame utilisé **une seule fois**, ou trop gros pour la mémoire (le cache évince d'autres données).
- **Alternative Lakehouse:** sauvegarder le résultat intermédiaire en **table Delta** (c'est exactement le rôle des couches **Bronze / Silver / Gold**).

## Conclusion — Questions clés

**1. Pourquoi `filter` est rapide et `groupBy` lent ?**
`filter` est une transformation **narrow**: chaque partition est traitée seule. `groupBy` est **wide**: il faut un **shuffle** (`Exchange`) pour regrouper les lignes de même clé sur la même partition (réseau + disque).

**2. Combien de jobs crée `df.count()` ?**
**Une action = un job.** Le nombre de **stages** dépend du nombre de shuffles dans le plan.

**3. Quand utiliser `repartition` vs `coalesce` ?**
`repartition` pour **augmenter** ou **équilibrer** les partitions (ou partitionner par clé) — avec shuffle. `coalesce` pour **réduire** le nombre de partitions sans shuffle (ex: avant une écriture).

**4. Quand le cache est-il utile ?**
Quand un DataFrame coûteux est **réutilisé plusieurs fois**. Inutile pour un usage unique. Pour partager un résultat entre notebooks / jobs, on le matérialise en **table Delta**.

**Leçons pour la suite du projet:**
- Filtrer **tôt** (Silver), agréger **tard** (Gold).
- Broadcaster la table `zones` dans les jointures.
- Matérialiser les étapes intermédiaires en **Delta** plutôt que `cache()`.